In [40]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file
file_path = "credit_card_transactions-ibm_v2.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "ealtman2019/credit-card-transactions",
  file_path,
)

In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
display("First 5 records:", df.head())

'First 5 records:'

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No


# Data preprocessing and Feature Engineering

In [42]:
# Lowering and standardizing all the column names
df.columns = (df.columns.str.strip().str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))

In [43]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24386900 entries, 0 to 24386899
Data columns (total 15 columns):
 #   Column          Dtype  
---  ------          -----  
 0   user            int64  
 1   card            int64  
 2   year            int64  
 3   month           int64  
 4   day             int64  
 5   time            object 
 6   amount          object 
 7   use_chip        object 
 8   merchant_name   int64  
 9   merchant_city   object 
 10  merchant_state  object 
 11  zip             float64
 12  mcc             int64  
 13  errors          object 
 14  is_fraud        object 
dtypes: float64(1), int64(7), object(7)
memory usage: 9.7 GB


In [44]:
# Fixing data types and reducing memory allocation for better performance
df["year"] = df["year"].astype("int16")
df["month"] = df["month"].astype("int8")
df["day"] = df["day"].astype("int8")
df["amount"] = df["amount"].str.replace("$", "", regex=False).astype("float")
df["merchant_name"] = df["merchant_name"].astype("object")
df["zip"] = df["zip"].astype("object")
df["mcc"] = df["mcc"].astype("int16")

In [45]:
# Crafting a timestamp column
df["timestamp"] = pd.to_datetime(df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2) + "-" + df["day"].astype(str).str.zfill(2) + " " + df["time"])

# Extracting 'hour' and 'day of week' columns using the timestamp column
df["hour"] = df["timestamp"].dt.hour.astype("int8")
df["dow"] = df["timestamp"].dt.dayofweek.astype("int8")

df["is_refund"] = df["amount"] < 0
df["is_online"] = df["merchant_city"].eq("ONLINE")
df["has_error"] = df["errors"].notna()

# Binning the 'target' column using mapping
df['is_fraud'] = df['is_fraud'].map({'Yes': 1, 'No': 0})

# Dropping the 'time' and 'errors' columns as we have already extracted the information
df = df.drop(columns=["time", "errors"]).sort_values(by="timestamp", kind="stable").reset_index(drop=True)

In [46]:
display("First 5 records:", df.head())

'First 5 records:'

,user,card,year,month,day,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,is_fraud,timestamp,hour,dow,is_refund,is_online,has_error
0,791,1,1991,1,2,68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,0,1991-01-02 07:10:00,7,2,False,False,False
1,791,1,1991,1,2,-68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,0,1991-01-02 07:17:00,7,2,True,False,False
2,791,1,1991,1,2,113.62,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,0,1991-01-02 07:21:00,7,2,False,False,False
3,791,1,1991,1,2,114.73,Swipe Transaction,-7269691894846892021,Burke,VA,22015.0,5411,0,1991-01-02 17:30:00,17,2,False,False,False
4,791,1,1991,1,3,251.71,Swipe Transaction,-3693650930986299431,Burke,VA,22015.0,4814,0,1991-01-03 09:03:00,9,3,False,False,False


In [47]:
# Observing memory usage and data types
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24386900 entries, 0 to 24386899
Data columns (total 19 columns):
 #   Column          Dtype         
---  ------          -----         
 0   user            int64         
 1   card            int64         
 2   year            int16         
 3   month           int8          
 4   day             int8          
 5   amount          float64       
 6   use_chip        object        
 7   merchant_name   object        
 8   merchant_city   object        
 9   merchant_state  object        
 10  zip             object        
 11  mcc             int16         
 12  is_fraud        int64         
 13  timestamp       datetime64[ns]
 14  hour            int8          
 15  dow             int8          
 16  is_refund       bool          
 17  is_online       bool          
 18  has_error       bool          
dtypes: bool(3), datetime64[ns](1), float64(1), int16(2), int64(3), int8(4), object(5)
memory usage: 6.8 GB


In [48]:
# Checking the shape of dataset
print(f"Total number of Rows: {df.shape[0]}")
print(f"Total number of Columns: {df.shape[1]}")

Total number of Rows: 24386900
Total number of Columns: 19


In [49]:
# Checking percentage of missing values
df.isna().mean().sort_values(ascending=False).mul(100).round(2)

zip               11.80
merchant_state    11.16
user               0.00
year               0.00
card               0.00
month              0.00
day                0.00
use_chip           0.00
amount             0.00
merchant_city      0.00
merchant_name      0.00
mcc                0.00
is_fraud           0.00
timestamp          0.00
hour               0.00
dow                0.00
is_refund          0.00
is_online          0.00
has_error          0.00
dtype: float64

In [50]:
# Checking for duplicate enteries
df[df.duplicated(keep=False)].sort_values(by='timestamp', ascending=True)

,user,card,year,month,day,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,is_fraud,timestamp,hour,dow,is_refund,is_online,has_error
317501,109,2,1999,12,9,120.0,Swipe Transaction,-4282466774399734331,Hammond,IN,46323.0,4829,0,1999-12-09 04:16:00,4,3,False,False,True
317502,109,2,1999,12,9,120.0,Swipe Transaction,-4282466774399734331,Hammond,IN,46323.0,4829,0,1999-12-09 04:16:00,4,3,False,False,True
575933,1963,0,2001,4,27,120.0,Swipe Transaction,-4282466774399734331,Lakeland,FL,33810.0,4829,0,2001-04-27 15:22:00,15,4,False,False,True
575934,1963,0,2001,4,27,120.0,Swipe Transaction,-4282466774399734331,Lakeland,FL,33810.0,4829,0,2001-04-27 15:22:00,15,4,False,False,True
956300,1032,1,2002,8,4,80.0,Swipe Transaction,-4282466774399734331,Louisville,KY,40222.0,4829,0,2002-08-04 14:11:00,14,6,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22823008,1160,3,2019,4,16,67.0,Chip Transaction,1799189980464955940,Round Rock,TX,78681.0,5499,0,2019-04-16 07:12:00,7,1,False,False,False
23034655,803,3,2019,5,31,80.0,Chip Transaction,-4282466774399734331,Riverside,CA,92508.0,4829,0,2019-05-31 07:16:00,7,4,False,False,True
23034656,803,3,2019,5,31,80.0,Chip Transaction,-4282466774399734331,Riverside,CA,92508.0,4829,0,2019-05-31 07:16:00,7,4,False,False,True
23835591,356,2,2019,11,16,120.0,Chip Transaction,-4282466774399734331,Perry,NY,14530.0,4829,0,2019-11-16 13:06:00,13,5,False,False,True


In [51]:
# Removing the duplicate rows
df = df.drop_duplicates()

In [52]:
# Extracting sequential features
df = df.sort_values(["user", "card", "timestamp"], kind="stable")
g = df.groupby(["user", "card"], observed=True)

# Time difference in seconds between the transactions
df["secs_since_prev"] = g["timestamp"].diff().dt.total_seconds()

# Rolling count of transactions for each user
df["prior_txn_count"] = g.cumcount()

# Rolling total of amount for each user
prior_sum = g["amount"].cumsum() - df["amount"]
df["prior_mean_amt"] = prior_sum / df["prior_txn_count"].replace(0, np.nan)

# Comparing the current transaction amount with average amount of all prior transactions 
df["amt_vs_prior_mean"] = (df["amount"] / df["prior_mean_amt"]).round(2)
# Replace infinity values that we may get after division with '0' with NaN
df["amt_vs_prior_mean"] = df["amt_vs_prior_mean"].replace([np.inf, -np.inf], np.nan)

df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)

In [53]:
display(df.head())

,user,card,year,month,day,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,is_fraud,timestamp,hour,dow,is_refund,is_online,has_error,secs_since_prev,prior_txn_count,prior_mean_amt,amt_vs_prior_mean
0,791,1,1991,1,2,68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,0,1991-01-02 07:10:00,7,2,False,False,False,NaN,0,NaN,NaN
1,791,1,1991,1,2,-68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,0,1991-01-02 07:17:00,7,2,True,False,False,420.0,1,68.000000,-1.00
2,791,1,1991,1,2,113.62,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,0,1991-01-02 07:21:00,7,2,False,False,False,240.0,2,0.000000,NaN
3,791,1,1991,1,2,114.73,Swipe Transaction,-7269691894846892021,Burke,VA,22015.0,5411,0,1991-01-02 17:30:00,17,2,False,False,False,36540.0,3,37.873333,3.03
4,791,1,1991,1,3,251.71,Swipe Transaction,-3693650930986299431,Burke,VA,22015.0,4814,0,1991-01-03 09:03:00,9,3,False,False,False,55980.0,4,57.087500,4.41
5,791,1,1991,1,3,16.28,Swipe Transaction,-7269691894846892021,Burke,VA,22015.0,5411,0,1991-01-03 11:14:00,11,3,False,False,False,7860.0,5,96.012000,0.17
6,791,1,1991,1,3,172.00,Swipe Transaction,3189517333335617109,Fairfax,VA,22030.0,5311,0,1991-01-03 12:46:00,12,3,False,False,False,5520.0,6,82.723333,2.08
7,791,1,1991,1,4,16.63,Swipe Transaction,5701841789931834110,Burke,VA,22015.0,5411,0,1991-01-04 11:09:00,11,4,False,False,False,80580.0,7,95.477143,0.17
8,791,1,1991,1,4,27.00,Swipe Transaction,-8194579483471190227,Burke,VA,22015.0,5211,0,1991-01-04 13:56:00,13,4,False,False,False,10020.0,8,85.621250,0.32
9,791,1,1991,1,5,210.64,Swipe Transaction,-3693650930986299431,Burke,VA,22015.0,4814,0,1991-01-05 08:40:00,8,5,False,False,False,67440.0,9,79.107778,2.66


In [56]:
# Taking stratified sample of 0.1 (10% of the total observations in the dataset)

FRAC = 0.10

eda = (df.groupby(["is_fraud", "year"], group_keys=False, observed=True)
         .sample(frac=FRAC, random_state=42)
         .sort_values("timestamp", kind="stable")
         .reset_index(drop=True))

print(f"Population observations: {len(df)}") 
print(f"Sample observations: {len(eda)}") 
print(f"Population fraud count: {df["is_fraud"].sum()}")
print(f"Sample fraud count: {eda["is_fraud"].sum()}")

Population observations: 24386833
Sample observations: 2438685
Population fraud count: 29757
Sample fraud count: 2976


In [62]:
# Validate that the sample matches the population

cmp = pd.DataFrame({
    "population":   [df["is_fraud"].mean(), df["amount"].mean(), df["amount"].median(), df["is_online"].mean()],
    "sample": [eda["is_fraud"].mean(), eda["amount"].mean(), eda["amount"].median(), eda["is_online"].mean()],
}, index=["fraud_rate", "mean_amt", "median_amt", "online_pct"]).round(3)

cmp["rel_diff_%"] = ((cmp["sample"] / cmp["population"] - 1) * 100).round(3)

print(cmp)

            population  sample  rel_diff_%
fraud_rate       0.001   0.001       0.000
mean_amt        43.634  43.624      -0.023
median_amt      30.140  30.120      -0.066
online_pct       0.112   0.112       0.000


For now onwards, the 'eda' (sample) dataframe will be used for plots and tests, and the 'df' dataframe will be used for population level aggregations/outcomes.

In [100]:
# Validating refunds and if any refund transactions were reported as fraud

print(f"Average amount of refund transactions: {(eda["amount"] <= 0).mean().round(3).item()}")
print(f"Refund transactions reported as fraudulent: {eda.loc[eda.is_refund, "is_fraud"].mean().round(3).item()}")

Average amount of refund transactions: 0.052
Refund transactions reported as fraudulent: 0.001


In [102]:
from statsmodels.stats.proportion import proportion_confint

n, k = len(df), int(df["is_fraud"].sum())
lo, hi = proportion_confint(k, n, method="wilson")
print(f"fraud rate {k/n:.4%}, 95% CI [{lo:.4%}, {hi:.4%}], imbalance 1:{(n-k)//k}")

fraud rate 0.1220%, 95% CI [0.1206%, 0.1234%], imbalance 1:818
